# Zero-shot H/T (Hate/Troll) detection on code-mixed Tulu
10 multilingual pretrained models, no fine-tuning. Metrics: Acc, Macro P/R/F1, Weighted-F1, H/T-class P/R/F1.

**Pure zero-shot** = fixed native threshold. **Calibrated** = threshold picked on the HT file, applied to test (uses labels from the other file, so report separately).

Runtime: Colab/Kaggle with GPU recommended (T4 is enough).

## 1. Install

In [1]:
!pip install -q torch transformers sentence-transformers sentencepiece accelerate scikit-learn pandas tabulate

## 2. Upload your CSV files
Run this cell and select `tulu__HT.csv` and `Test_Tulu_with_label.csv` (or skip if already in the working directory / mounted).

In [2]:
try:
    from google.colab import files
    import os
    if not (os.path.exists('/kaggle/input/datasets/hegdeasha/tulu-ieee/tulu__HT.csv') and os.path.exists('/kaggle/input/datasets/hegdeasha/tulu-ieee/Test_Tulu_with_label.csv')):
        files.upload()
except ImportError:
    pass  # not on Colab; place the CSVs next to this notebook

HT_PATH = '/kaggle/input/datasets/hegdeasha/tulu-ieee/tulu__HT.csv'
TEST_PATH = '/kaggle/input/datasets/hegdeasha/tulu-ieee/Test_Tulu_with_label.csv'

## 3. Imports and configuration

In [3]:
import argparse, gc, time, traceback
import numpy as np, pandas as pd
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score)

POS, NEG = "H/T", "NON H/T"
BATCH = 16
MAXLEN = 128

## label descriptions

In [4]:
POS_DESC = [
    "A hateful, abusive, offensive or trolling comment.",
    "A comment that insults, mocks or attacks a person or a group.",
    "A toxic comment containing hate speech or sarcasm meant to troll.",
]
NEG_DESC = [
    "A neutral, polite or informative comment.",
    "A comment that appreciates, supports or asks a genuine question.",
    "A harmless comment with no hate, abuse or trolling.",
]
YN_PROMPT = ("The following YouTube comment is written in Tulu mixed with Kannada/English/Malayalam "
             "in Roman script. Is it hateful, abusive, offensive or trolling? Answer Yes or No.\n"
             "Comment: {text}")

## model registry

In [5]:
# key: (hub id, approach, extra)
REGISTRY = {
    "xlmr_xnli":   ("joeddav/xlm-roberta-large-xnli",                         "nli", {}),
    "mdeberta":    ("MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7", "nli", {}),
    "labse":       ("sentence-transformers/LaBSE",                            "emb", {}),
    "mpnet_multi": ("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "emb", {}),
    "me5_large":   ("intfloat/multilingual-e5-large",                         "emb", {"prefix": "query: "}),
    "bge_m3":      ("BAAI/bge-m3",                                            "emb", {}),
    "xlmr_toxic":  ("textdetox/xlmr-large-toxicity-classifier",               "cls", {}),
    "mt0":         ("bigscience/mt0-large",                                   "seq2seq", {}),
    "bloomz":      ("bigscience/bloomz-3b",                                   "causal", {"yes": " Yes", "no": " No", "chat": False}),
    "qwen":        ("Qwen/Qwen2.5-3B-Instruct",                               "causal", {"yes": "Yes", "no": "No", "chat": True}),
}


def free():
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


def batches(xs, n=BATCH):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]

## scorers: return (score, native_threshold)

In [7]:
# score is higher => more likely H/T.
def score_nli(model_id, texts, **_):
    import torch
    from transformers import pipeline
    dev = 0 if torch.cuda.is_available() else -1
    pipe = pipeline("zero-shot-classification", model=model_id, device=dev)
    labels = ["hateful, abusive or trolling", "neutral or appreciative"]
    out = pipe(texts, candidate_labels=labels, hypothesis_template="This comment is {}.",
               batch_size=BATCH, truncation=True)
    s = [o["scores"][o["labels"].index(labels[0])] for o in out]
    del pipe
    return np.array(s), 0.5


def score_emb(model_id, texts, prefix="", **_):
    import torch
    from sentence_transformers import SentenceTransformer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m = SentenceTransformer(model_id, device=dev)
    m.max_seq_length = MAXLEN
    enc = lambda xs: m.encode([prefix + x for x in xs], batch_size=BATCH, normalize_embeddings=True,
                              show_progress_bar=False)
    x = enc(texts)
    p = enc(POS_DESC).mean(0); n = enc(NEG_DESC).mean(0)
    p /= np.linalg.norm(p); n /= np.linalg.norm(n)
    del m
    return x @ p - x @ n, 0.0


def score_cls(model_id, texts, **_):
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_id)
    m = AutoModelForSequenceClassification.from_pretrained(model_id).to(dev).eval()
    id2 = {int(k): v.lower() for k, v in m.config.id2label.items()}
    pos_idx = next((k for k, v in id2.items() if "toxic" in v and not v.startswith(("non", "not", "un"))), 1)
    print("   label map:", id2, "-> positive idx", pos_idx)
    sc = []
    with torch.no_grad():
        for b in batches(texts):
            t = tok(b, return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            sc += torch.softmax(m(**t).logits.float(), -1)[:, pos_idx].cpu().tolist()
    del m
    return np.array(sc), 0.5


def score_seq2seq(model_id, texts, **_):
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    dt = torch.bfloat16 if (dev == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
    tok = AutoTokenizer.from_pretrained(model_id)
    m = AutoModelForSeq2SeqLM.from_pretrained(model_id, torch_dtype=dt).to(dev).eval()
    yes = tok("Yes", add_special_tokens=False).input_ids[0]
    no = tok("No", add_special_tokens=False).input_ids[0]
    sc = []
    with torch.no_grad():
        for b in batches(texts):
            t = tok([YN_PROMPT.format(text=x) for x in b], return_tensors="pt", padding=True,
                    truncation=True, max_length=MAXLEN + 64).to(dev)
            dec = torch.full((len(b), 1), m.config.decoder_start_token_id, device=dev)
            lg = m(**t, decoder_input_ids=dec).logits[:, 0, :].float()
            sc += (lg[:, yes] - lg[:, no]).cpu().tolist()
    del m
    return np.array(sc), 0.0


def score_causal(model_id, texts, yes=" Yes", no=" No", chat=False, **_):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    dt = torch.float16 if dev == "cuda" else torch.float32
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dt).to(dev).eval()
    y = tok.encode(yes, add_special_tokens=False)[0]
    n = tok.encode(no, add_special_tokens=False)[0]

    def build(x):
        p = YN_PROMPT.format(text=x)
        if chat:
            return tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                           add_generation_prompt=True)
        return p + "\nAnswer:"

    sc = []
    with torch.no_grad():
        for b in batches(texts, 8):
            t = tok([build(x) for x in b], return_tensors="pt", padding=True, truncation=True,
                    max_length=MAXLEN + 128).to(dev)
            lg = m(**t).logits[:, -1, :].float()
            sc += (lg[:, y] - lg[:, n]).cpu().tolist()
    del m
    return np.array(sc), 0.0


SCORERS = {"nli": score_nli, "emb": score_emb, "cls": score_cls, "seq2seq": score_seq2seq, "causal": score_causal}

## metrics

In [8]:
def metrics(y, pred):
    pm, rm, fm, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    pp, rp, fp, _ = precision_recall_fscore_support(y, pred, labels=[1], zero_division=0)
    return {"Acc": accuracy_score(y, pred), "Macro-P": pm, "Macro-R": rm, "Macro-F1": fm,
            "Weighted-F1": f1_score(y, pred, average="weighted", zero_division=0),
            "H/T-P": pp[0], "H/T-R": rp[0], "H/T-F1": fp[0]}


def best_threshold(y, s):
    cands = np.unique(np.quantile(s, np.linspace(0.02, 0.98, 97)))
    return max(cands, key=lambda t: f1_score(y, (s >= t).astype(int), average="macro", zero_division=0))


def load(path):
    df = pd.read_csv(path).dropna(subset=["Text", "Label"])
    df["Text"] = df["Text"].astype(str).str.strip()
    df["y"] = (df["Label"].str.strip() == POS).astype(int)
    return df

## Run the benchmark
Pick models by key (see `REGISTRY`). Set `DRY = True` for a quick smoke test with random scores (no downloads; numbers are meaningless).

In [9]:
MODELS = "all"      # or e.g. "labse,mt0,bloomz"
DRY = False

ht, te = load(HT_PATH), load(TEST_PATH)
print(len(ht), "HT rows |", len(te), "test rows")
keys = list(REGISTRY) if MODELS == "all" else MODELS.split(",")
rows = []
rng = np.random.default_rng(0)

for k in keys:
    mid, kind, extra = REGISTRY[k]
    print(f"\n=== {k}: {mid} [{kind}]")
    t0 = time.time()
    try:
        if DRY:
            s_ht, s_te, thr = rng.normal(size=len(ht)), rng.normal(size=len(te)), 0.0
        else:
            s_all, thr = SCORERS[kind](mid, list(ht.Text) + list(te.Text), **extra)
            s_ht, s_te = s_all[:len(ht)], s_all[len(ht):]
    except Exception:
        traceback.print_exc()
        print(f"   !! {k} failed, skipping")
        free(); continue
    free()
    cal = best_threshold(ht.y.values, s_ht)
    for name, df, s in (("HT file", ht, s_ht), ("Test file", te, s_te)):
        r = {"Model": mid, "Approach": kind, "Dataset": name, "N": len(df)}
        r.update(metrics(df.y.values, (s >= thr).astype(int)))
        r["Macro-F1 (calibrated on HT)"] = (f1_score(df.y.values, (s >= cal).astype(int), average="macro")
                                             if name == "Test file" else np.nan)
        rows.append(r)
    pd.DataFrame({"Text": te.Text, "Label": te.Label, "score": s_te,
                  "pred": np.where(s_te >= thr, POS, NEG)}).to_csv(f"predictions_{k}.csv", index=False)
    print(f"   done in {time.time()-t0:.0f}s")

res = pd.DataFrame(rows)
res.to_csv("results.csv", index=False)

730 HT rows | 379 test rows

=== xlmr_xnli: joeddav/xlm-roberta-large-xnli [nli]


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

   done in 106s


## Results table

In [10]:
pd.set_option("display.width", 250)
cols = ["Model","Approach","Acc","Macro-P","Macro-R","Macro-F1","Weighted-F1","H/T-P","H/T-R","H/T-F1"]
print("Baseline (always NON H/T): HT Macro-F1 = %.3f | Test Macro-F1 = %.3f" % (
    f1_score(ht.y, 0*ht.y, average="macro"), f1_score(te.y, 0*te.y, average="macro")))
print("\n=== TEST FILE ===")
display(res[res.Dataset=="Test file"][cols + ["Macro-F1 (calibrated on HT)"]].round(3).sort_values("Macro-F1", ascending=False).reset_index(drop=True))
print("\n=== HT FILE ===")
display(res[res.Dataset=="HT file"][cols].round(3).sort_values("Macro-F1", ascending=False).reset_index(drop=True))

Baseline (always NON H/T): HT Macro-F1 = 0.426 | Test Macro-F1 = 0.452

=== TEST FILE ===


,Model,Approach,Acc,Macro-P,Macro-R,Macro-F1,Weighted-F1,H/T-P,H/T-R,H/T-F1,Macro-F1 (calibrated on HT)
0,BAAI/bge-m3,emb,0.778,0.602,0.590,0.595,0.771,0.351,0.299,0.323,0.595
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,nli,0.828,0.695,0.579,0.594,0.794,0.542,0.194,0.286,0.639
2,sentence-transformers/paraphrase-multilingual-...,emb,0.776,0.582,0.565,0.570,0.762,0.320,0.239,0.274,0.625
3,textdetox/xlmr-large-toxicity-classifier,cls,0.807,0.621,0.561,0.568,0.776,0.400,0.179,0.247,0.593
4,sentence-transformers/LaBSE,emb,0.704,0.558,0.574,0.560,0.723,0.263,0.373,0.309,0.567
5,intfloat/multilingual-e5-large,emb,0.826,0.701,0.525,0.506,0.763,0.571,0.060,0.108,0.590
6,joeddav/xlm-roberta-large-xnli,nli,0.826,0.747,0.513,0.481,0.754,0.667,0.030,0.057,0.534
7,Qwen/Qwen2.5-3B-Instruct,causal,0.826,0.913,0.507,0.467,0.750,1.000,0.015,0.029,0.619
8,bigscience/bloomz-3b,causal,0.813,0.495,0.499,0.462,0.743,0.167,0.015,0.027,0.510
9,bigscience/mt0-large,seq2seq,0.805,0.466,0.495,0.459,0.739,0.111,0.015,0.026,0.350



=== HT FILE ===


,Model,Approach,Acc,Macro-P,Macro-R,Macro-F1,Weighted-F1,H/T-P,H/T-R,H/T-F1
0,sentence-transformers/LaBSE,emb,0.656,0.538,0.536,0.536,0.651,0.316,0.287,0.301
1,BAAI/bge-m3,emb,0.671,0.535,0.528,0.528,0.654,0.314,0.234,0.268
2,textdetox/xlmr-large-toxicity-classifier,cls,0.686,0.534,0.523,0.518,0.656,0.315,0.186,0.234
3,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,nli,0.701,0.541,0.523,0.513,0.660,0.330,0.154,0.210
4,sentence-transformers/paraphrase-multilingual-...,emb,0.684,0.522,0.514,0.507,0.650,0.295,0.165,0.212
5,intfloat/multilingual-e5-large,emb,0.758,0.760,0.538,0.506,0.677,0.762,0.085,0.153
6,Qwen/Qwen2.5-3B-Instruct,causal,0.747,0.873,0.508,0.443,0.642,1.000,0.016,0.031
7,bigscience/mt0-large,seq2seq,0.734,0.471,0.498,0.433,0.634,0.200,0.011,0.020
8,bigscience/bloomz-3b,causal,0.734,0.471,0.498,0.433,0.634,0.200,0.011,0.020
9,joeddav/xlm-roberta-large-xnli,nli,0.736,0.442,0.497,0.429,0.632,0.143,0.005,0.010


In [13]:
# Optional: download outputs on Colab
try:
    from google.colab import files
    files.download("results.csv")
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>